# Plot multi-lead predictions for predicting hurricane track (distance) errors.
author: Elizabeth A. Barnes and Randal J. Barnes

In [1]:
%matplotlib inline
%load_ext autotime

import sys
import importlib as imp
import warnings
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy as ct
import plots
import compute_predictions

import experiment_settings
import mahalanobis

time: 1.75 s (started: 2022-12-16 12:28:49 -07:00)


In [2]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

DATA_PATH = "data/"
MODEL_PATH = "saved_models/"
FIGURE_PATH = "figures/analysis/"
PREDICTIONS_PATH = "saved_predictions/"

time: 261 µs (started: 2022-12-16 12:28:51 -07:00)


In [3]:
plt.style.use("seaborn-white")
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 600
plots.set_plot_rc()
np.warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

time: 465 µs (started: 2022-12-16 12:28:51 -07:00)


In [4]:
EXP_NAME_VEC = (
    # "centered_bivariate_normal_000_EPCP24",

    "centered_bivariate_normal_100_EPCP12",
    "centered_bivariate_normal_101_EPCP24",
    "centered_bivariate_normal_102_EPCP36",
    "centered_bivariate_normal_103_EPCP48",
    "centered_bivariate_normal_104_EPCP60",
    "centered_bivariate_normal_105_EPCP72",
    "centered_bivariate_normal_106_EPCP84",
    "centered_bivariate_normal_107_EPCP96",
    "centered_bivariate_normal_108_EPCP108",
    "centered_bivariate_normal_109_EPCP120",

    "centered_bivariate_normal_200_AL12",
    "centered_bivariate_normal_201_AL24",
    "centered_bivariate_normal_202_AL36",
    "centered_bivariate_normal_203_AL48",
    "centered_bivariate_normal_204_AL60",
    "centered_bivariate_normal_205_AL72",
    "centered_bivariate_normal_206_AL84",
    "centered_bivariate_normal_207_AL96",
    "centered_bivariate_normal_208_AL108",
    "centered_bivariate_normal_209_AL120",

    )


time: 935 µs (started: 2022-12-16 12:28:51 -07:00)


# Plot Results

In [9]:
storm_dict = {
    "IAN": {"storm_name": "IAN",
            "year": 2022,
            "extent": [-100,-65,5,35],
            "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
            # https://www.nhc.noaa.gov/aboutcone.shtml
            },
    "FIONA": {"storm_name": "FIONA",
              "year": 2022,
              "extent": [-100,-45,10,55],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "IRMA": {"storm_name": "IRMA",
             "year": 2017,
             "extent": [-100,-20,10,40],
             "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
             # https://www.air-worldwide.com/blog/posts/2017/8/the-ever-shrinking-cone-of-uncertainty/
             },
    "NICOLE": {"storm_name": "NICOLE",
               "year": 2022,
               "extent": [-100,-48,20,45],
               "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
               },
    "JULIA": {"storm_name": "JULIA",
              "year": 2022,
              "extent": [-100,-65,5,20],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "NORMAN": {"storm_name": "NORMAN",
                "year": 2018,
                "pred_time": 90306,
                "extent": [195, 360-135, 5, 35],
                "nhc_cone_radius": {0:8, 12:25, 24:40, 36:51, 48:66, 60:93, 72:93, 96:116, 120:151}
                },
}

time: 817 µs (started: 2022-12-16 13:12:38 -07:00)


In [10]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)

for storm_name in ("FIONA",):#("IAN","IRMA","FIONA"):#("IAN", "NICOLE", "IRMA"):
    print(storm_name)
    storm = storm_dict[storm_name]

    for RNG_SEED in (1,2,3):

        TESTING_YEAR = storm["year"]

        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for exp_name in EXP_NAME_VEC:
            settings = experiment_settings.get_settings(exp_name)

            # Create the model name.
            model_name = (
                    exp_name
                    + "_"
                    + str(TESTING_YEAR)
                    + "_"
                    + settings["uncertainty_type"]
                    + "_"
                    + f"rng_seed_{RNG_SEED}"
            )
            try:
                prediction_filename = PREDICTIONS_PATH + model_name + "_testing_predictions.csv"
                df = pd.read_csv(prediction_filename)
            except:
                continue

            settings["rng_seed"] = RNG_SEED
            settings["years_test"] = (TESTING_YEAR,)
            df["exp_name"] = exp_name
            df_pred_test = pd.concat([df_pred_test, df], axis=0)

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["Name"] == storm["storm_name"])].copy()
        df = df.sort_values("time").reset_index(drop=True)
        forecast_dates = df["time"].unique()

        for i,pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            storm["pred_time"] = pred_time
            df_storm = df_pred_test.loc[
                (df_pred_test["Name"] == storm["storm_name"]) & (df_pred_test["time"] == storm["pred_time"])].copy()
            df_storm = compute_predictions.add_lead_zero(df_storm)
            df_storm["nhc_cone_radius"] = [storm["nhc_cone_radius"][key] for key in df_storm["ftime(hr)"].unique()]


            # plot probability ellipses
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=np.arange(0,120+12, 12),
                contours=(.1, .25, .5, .75, .9,),
                extent = storm["extent"],
                alpha=.4,
                vector=True,
            )
            plt.gca().get_legend().remove()
            ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + 'probability_ellipses_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()


            # plot banana cones
            try:
                fig = plt.figure(dpi=150, )
                ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
                details = plots.plot_banana_of_uncertainty(
                    df_storm=df_storm,
                    ax=ax,
                    extent=storm["extent"],
                    vector=True,
                    colors=("steelblue","khaki"),
                    alpha=.75,
                    plot_nhc_cone=True,
                )
                ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
                plt.savefig(
                    FIGURE_PATH + 'banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                    dpi=dpiFig,
                    bbox_inches='tight',
                )
                plt.close()
            except:
                print('not enough data for spline computation. not making the figure.')
                plt.close()


FIONA
1 of 28: 91412
2 of 28: 91418
3 of 28: 91500
4 of 28: 91506
5 of 28: 91512
6 of 28: 91518
7 of 28: 91600
8 of 28: 91606
9 of 28: 91612
10 of 28: 91618
11 of 28: 91700
12 of 28: 91706
13 of 28: 91712
14 of 28: 91718
15 of 28: 91800
16 of 28: 91806
17 of 28: 91812
18 of 28: 91818
19 of 28: 91900
20 of 28: 91906
21 of 28: 91912
22 of 28: 91918
23 of 28: 92000
24 of 28: 92006
25 of 28: 92012
26 of 28: 92018
27 of 28: 92100
28 of 28: 92106
1 of 28: 91412
2 of 28: 91418
3 of 28: 91500
4 of 28: 91506
5 of 28: 91512
6 of 28: 91518
7 of 28: 91600
8 of 28: 91606
9 of 28: 91612
10 of 28: 91618
11 of 28: 91700
12 of 28: 91706
13 of 28: 91712
14 of 28: 91718
15 of 28: 91800
16 of 28: 91806
17 of 28: 91812
18 of 28: 91818
19 of 28: 91900
20 of 28: 91906
21 of 28: 91912
22 of 28: 91918
23 of 28: 92000
24 of 28: 92006
25 of 28: 92012
26 of 28: 92018
27 of 28: 92100
28 of 28: 92106
1 of 28: 91412
2 of 28: 91418
3 of 28: 91500
4 of 28: 91506
5 of 28: 91512
6 of 28: 91518
7 of 28: 91600
8 of 28: 91

time: 37min 31s (started: 2022-12-16 12:28:51 -07:00)
